# Notebook 2: YOLOv8 Fine-tuning on BDD100K
**Author:** David Ho

> **Prerequisite:** Run `01_dataset_preparation.ipynb` once before this.

## Step 1: Mount Drive + Check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — go to Runtime > Change runtime type > T4 GPU')

## Step 2: Install Dependencies

In [ ]:
!pip install -q ultralytics

In [ ]:
import json, time, random, zipfile, shutil
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from ultralytics import YOLO
random.seed(42)

## Step 3: Config

In [ ]:
DRIVE_ROOT    = Path('/content/drive/MyDrive/vehicle_detection')
DRIVE_BACKUP  = DRIVE_ROOT / 'processed'
DRIVE_RESULTS = DRIVE_ROOT / 'results' / 'yolo'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

LOCAL_BDD  = Path('/content/bdd100k')
LOCAL_DATA = Path('/content/data')

VEHICLE_CLASSES = ['car', 'truck', 'bus', 'motorcycle']
MODEL_VARIANT   = 'yolov8s.pt'
EPOCHS          = 50
IMG_SIZE        = 640
BATCH_SIZE      = 16
LR0             = 0.01
DEVICE          = 0 if torch.cuda.is_available() else 'cpu'

assert DRIVE_BACKUP.exists(), 'Run Notebook 1 first.'
print('Config OK')

## Step 4: Session Setup
Unzips images locally (fast) and restores label files from Drive. Run this every new Colab session.

In [ ]:
DRIVE_ZIP = DRIVE_ROOT / 'downloads' / 'solesensei_bdd100k.zip'

# 1. Unzip images to local if not already done this session
if not LOCAL_BDD.exists():
    print('Unzipping BDD100K to local disk (~5 min)...')
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
        z.extractall('/content/')
    print('Unzip done.')

# Auto-detect extra nesting
if not (LOCAL_BDD / 'images').exists() and (LOCAL_BDD / 'bdd100k' / 'images').exists():
    LOCAL_BDD = LOCAL_BDD / 'bdd100k'

# 2. Restore label files from Drive (~1 min)
if not LOCAL_DATA.exists():
    print('Restoring label files from Drive...')
    shutil.copytree(DRIVE_BACKUP, LOCAL_DATA)
    print('Labels restored.')

# 3. Fix dataset.yaml to use current LOCAL_BDD path
yaml_path = LOCAL_DATA / 'yolo' / 'dataset.yaml'
yaml_txt = f"""path: {LOCAL_BDD}
train: images/100k/train
val:   images/100k/val
test:  images/100k/val

nc: {len(VEHICLE_CLASSES)}
names: {VEHICLE_CLASSES}
"""
yaml_path.write_text(yaml_txt)

# 4. Tell Ultralytics where labels are (must match images/ -> labels/ convention)
# Symlink: LOCAL_BDD/labels/100k -> LOCAL_DATA/yolo/labels
lbl_target = LOCAL_BDD / 'labels' / '100k'
lbl_target.parent.mkdir(parents=True, exist_ok=True)
if not lbl_target.exists():
    lbl_target.symlink_to(LOCAL_DATA / 'yolo' / 'labels')

print('Session setup complete.')
print('  Images :', LOCAL_BDD / 'images' / '100k')
print('  Labels :', LOCAL_BDD / 'labels' / '100k')
print('  YAML   :', yaml_path)

## Step 5: Model Overview

In [ ]:
model = YOLO(MODEL_VARIANT)
total_params     = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
print(f'Model            : {MODEL_VARIANT}')
print(f'Total params     : {total_params:,}')
print(f'Trainable params : {trainable_params:,}')

## Step 6: Train

In [ ]:
t0 = time.time()

model.train(
    data     = str(yaml_path),
    epochs   = EPOCHS,
    imgsz    = IMG_SIZE,
    batch    = BATCH_SIZE,
    lr0      = LR0,
    device   = DEVICE,
    project  = str(DRIVE_RESULTS),   # checkpoints saved directly to Drive
    name     = 'train',
    exist_ok = True,
    patience = 15,
    save     = True,
    plots    = True,
    fliplr   = 0.5,
    hsv_h    = 0.015,
    hsv_s    = 0.7,
    hsv_v    = 0.4,
    mosaic   = 1.0,
)

train_time = time.time() - t0
print(f'Total time  : {train_time/60:.1f} min')
print(f'Per epoch   : {train_time/EPOCHS:.1f} s')

timing = {'model': 'YOLOv8s', 'total_train_time_s': round(train_time, 2),
          'time_per_epoch_s': round(train_time/EPOCHS, 2), 'epochs': EPOCHS}
(DRIVE_RESULTS / 'timing.json').write_text(json.dumps(timing, indent=2))

## Step 7: Training Curves

In [ ]:
df = pd.read_csv(DRIVE_RESULTS / 'train' / 'results.csv')
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('YOLOv8s Training Curves', fontsize=14)
metrics = [
    ('train/box_loss',       'val/box_loss',         'Box Loss'),
    ('train/cls_loss',       'val/cls_loss',         'Class Loss'),
    ('train/dfl_loss',       'val/dfl_loss',         'DFL Loss'),
    ('metrics/precision(B)', None,                   'Precision'),
    ('metrics/recall(B)',    None,                   'Recall'),
    ('metrics/mAP50(B)',     'metrics/mAP50-95(B)', 'mAP'),
]
for ax, (c1, c2, title) in zip(axes.flat, metrics):
    if c1 in df.columns: ax.plot(df['epoch'], df[c1], label='train' if c2 else c1.split('/')[-1])
    if c2 and c2 in df.columns: ax.plot(df['epoch'], df[c2], label=c2.split('/')[-1], linestyle='--')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(str(DRIVE_RESULTS / 'yolo_training_curves.png'), dpi=120)
plt.show()

## Step 8: Evaluate on Test Set

In [ ]:
model_best  = YOLO(str(DRIVE_RESULTS / 'train' / 'weights' / 'best.pt'))
val_metrics = model_best.val(data=str(yaml_path), split='test',
                              imgsz=IMG_SIZE, device=DEVICE, verbose=True)

map50   = val_metrics.box.map50
map5095 = val_metrics.box.map
prec    = val_metrics.box.mp
rec     = val_metrics.box.mr
f1      = 2 * prec * rec / (prec + rec + 1e-9)

print(f'mAP@50    : {map50:.4f}')
print(f'mAP@50-95 : {map5095:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1-score  : {f1:.4f}')

print('\nPer-class mAP@50:')
for cls_id, ap in zip(val_metrics.box.ap_class_index, val_metrics.box.ap50):
    print(f'  {VEHICLE_CLASSES[cls_id]:12s}: {ap:.4f}')

## Step 9: Inference Speed

In [ ]:
# Use only test-split images for the benchmark
split_names = json.loads((LOCAL_DATA / 'yolo' / 'split_names.json').read_text())
test_names  = set(split_names['test'])
test_imgs   = [p for p in (LOCAL_BDD / 'images' / '100k' / 'val').glob('*.jpg')
               if p.name in test_names][:100]

model_best.predict(str(test_imgs[0]), imgsz=IMG_SIZE, device=DEVICE, verbose=False)  # warmup

t0 = time.time()
for p in test_imgs:
    model_best.predict(str(p), imgsz=IMG_SIZE, device=DEVICE, verbose=False)
elapsed = time.time() - t0

ms_per_img = elapsed / len(test_imgs) * 1000
fps        = 1000 / ms_per_img
print(f'Inference : {ms_per_img:.1f} ms/image  ({fps:.1f} FPS)')

In [ ]:
metrics_out = {
    'model': 'YOLOv8s',
    'map50': round(float(map50), 4), 'map50_95': round(float(map5095), 4),
    'precision': round(float(prec), 4), 'recall': round(float(rec), 4),
    'f1': round(float(f1), 4), 'inference_ms': round(ms_per_img, 2),
    'fps': round(fps, 1), 'trainable_params': trainable_params,
    'time_per_epoch_s': timing['time_per_epoch_s'],
}
(DRIVE_RESULTS / 'metrics.json').write_text(json.dumps(metrics_out, indent=2))
print('Saved metrics.json')
print(json.dumps(metrics_out, indent=2))

## Step 10: Sample Detections

In [ ]:
sample = random.sample(test_imgs, min(6, len(test_imgs)))
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
fig.suptitle('YOLOv8s — Sample Detections on Test Set', fontsize=13)
COLORS = {0: '#2196F3', 1: '#FF9800', 2: '#4CAF50', 3: '#E91E63'}

for ax, img_path in zip(axes.flat, sample):
    result = model_best.predict(str(img_path), imgsz=IMG_SIZE, device=DEVICE,
                                conf=0.25, verbose=False)[0]
    ax.imshow(Image.open(img_path))
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cls = int(box.cls[0])
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1,
                              linewidth=2, edgecolor=COLORS[cls], facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-4, f"{VEHICLE_CLASSES[cls]} {float(box.conf[0]):.2f}",
                color=COLORS[cls], fontsize=7, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig(str(DRIVE_RESULTS / 'yolo_sample_detections.png'), dpi=120)
plt.show()
print('YOLOv8 training complete.')